# Task 1 - Smart Buildings: Dataset Understanding and Probability

**Member 1 - Data Lead**

> Member 5: replace this starter with the final problem statement, objectives, and roadmap before submission.

This notebook analyzes the Smart Buildings generator dataset. It documents the source files, defines a reproducible analysis frame, checks data quality, and publishes shared event columns for the probability applications. Later sections are reserved for the team. Results describe this generator's accepted outputs, not real-world buildings.

## 1. Setup (Member 1)

Load the three source CSVs without editing them. The notebook first checks its current folder and the usual shared Drive folder. If the files are not there and this is Colab, it asks you to upload the three CSVs. If running locally, keep this notebook beside the CSVs or set `DATA_DIR` to their folder.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

REQUIRED_FILES = ['buildings_200_features.csv', 'rejected_seeds.csv', 'feature_catalog.csv']
candidate_dirs = [Path.cwd(), Path('/content/drive/MyDrive/stats_project')]
DATA_DIR = next((p for p in candidate_dirs if all((p / f).exists() for f in REQUIRED_FILES)), None)

if DATA_DIR is None:
    try:
        from google.colab import files
        print('Upload buildings_200_features.csv, rejected_seeds.csv, and feature_catalog.csv')
        files.upload()
        DATA_DIR = Path.cwd()
    except ImportError:
        raise FileNotFoundError(
            'Could not find the three CSV files. Put them beside this notebook or change DATA_DIR to their folder.'
        )

missing_files = [f for f in REQUIRED_FILES if not (DATA_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing {missing_files} in {DATA_DIR}. Upload all three project CSV files.')

df_raw = pd.read_csv(DATA_DIR / 'buildings_200_features.csv')
rej_raw = pd.read_csv(DATA_DIR / 'rejected_seeds.csv')
cat_raw = pd.read_csv(DATA_DIR / 'feature_catalog.csv')

print('Data folder:', DATA_DIR)
print('buildings:', df_raw.shape)
print('rejected seeds:', rej_raw.shape)
print('feature catalog:', cat_raw.shape)
assert df_raw.shape == (200, 54)
assert rej_raw.shape == (82, 2)
assert cat_raw.shape == (133, 4)

## 2. Understand the files and variables (Member 1)

Each building row is one accepted generated building configuration. Its seed is the link key. The rejected-seed file records generator attempts that failed the minimum zone-area rule. Together, the 200 accepted and 82 rejected seeds make the complete contiguous run from 100000 through 100281. The feature catalog documents all 54 CSV columns and 79 additional observation features available only during simulation.

| File | Shape | One row means | Link key / use |
|---|---:|---|---|
| `buildings_200_features.csv` | 200 x 54 | One accepted generated building | `cfg__seed` (also recorded as `meta__seed_requested`) |
| `rejected_seeds.csv` | 82 x 2 | One refused generator seed and its reason | `seed` |
| `feature_catalog.csv` | 133 x 4 | One feature definition | `column`; 54 are in the CSV and 79 are simulation-only |

Groups: `meta` records provenance, `cfg` records static configuration, `derived` records structure measured from generated files, and `obs` records observation-schema counts/derived vector length. The observation values themselves are not stored per building in this CSV.

In [ ]:
# A small readable sample: selected columns cover identifiers, configuration, structure, and schema.
sample_cols = [
    'meta__seed_requested', 'cfg__name', 'cfg__floor_plan.building_shape',
    'cfg__floor_plan.num_floors', 'cfg__ahu.num_ahus', 'derived__n_zones',
    'derived__ahus_added_beyond_config', 'cfg__simulation.occupants_per_zone',
    'obs__expected_vector_length_derived'
]
df_raw.sample(5, random_state=42)[sample_cols]

### Variable dictionary

The next table includes every CSV field. `Meaning` is a plain-English guide inferred from the feature name and the catalog's group/kind; the catalog remains the authoritative source for the feature definition.

In [ ]:
def classify_type(column, series):
    name = column.split('__', 1)[-1]
    if 'timestamp' in name or 'date' in name or 'time' in name:
        return 'date/time'
    if pd.api.types.is_numeric_dtype(series):
        if pd.api.types.is_integer_dtype(series):
            return 'numeric-discrete'
        return 'numeric-continuous'
    if any(token in name for token in ('name', 'sha256', 'output_files', 'shape')):
        return 'text/identifier'
    return 'categorical'

def plain_meaning(column):
    n = column.split('__', 1)[-1]
    meanings = {
        'seed_requested':'Seed requested by the generator', 'seed':'Seed saved in the building configuration',
        'weather_preset':'Weather preset used for scenario generation', 'generator_output_files':'Generated file names',
        'arrays_sha256':'Hash used to identify generated array contents', 'name':'Generated building name',
        'ahu.num_ahus':'Configured number of air handlers', 'ahu.max_zones_per_ahu':'Maximum zones per air handler',
        'floor_plan.building_shape':'Footprint shape family', 'floor_plan.composite_coverage':'Share of footprint covered by composite geometry',
        'floor_plan.composite_style':'Composite footprint style', 'floor_plan.height':'Building height parameter',
        'floor_plan.max_room_size':'Configured maximum room size', 'floor_plan.min_room_size':'Configured minimum room size',
        'floor_plan.num_floors':'Configured number of floors', 'floor_plan.shape_ratio':'Footprint width-to-height shape parameter',
        'floor_plan.width':'Footprint width parameter', 'materials.exterior_cv_conductivity':'Exterior construction conductivity parameter',
        'materials.exterior_cv_density':'Exterior construction density parameter', 'materials.exterior_cv_heat_capacity':'Exterior construction heat-capacity parameter',
        'materials.interior_mass_cv_density':'Interior mass density parameter', 'materials.interior_wall_cv_conductivity':'Interior wall conductivity parameter',
        'materials.interior_wall_cv_density':'Interior wall density parameter', 'simulation.num_days_in_episode':'Simulation episode length in days',
        'simulation.occupants_per_zone':'Configured occupants per zone', 'simulation.start_timestamp':'Simulation start time',
        'n_zones':'Number of generated thermal zones', 'n_vav_boxes':'Number of VAV boxes', 'n_ahus':'Number of air handlers in generated scenario',
        'ahus_configured':'Configured AHU count', 'ahus_added_beyond_config':'AHUs added beyond configured count',
        'max_zones_per_ahu':'Maximum zones assigned to an AHU', 'num_floors':'Floor count measured from generated structure',
        'max_rooms_on_one_ahu':'Maximum rooms assigned to one AHU', 'min_rooms_on_one_ahu':'Minimum rooms assigned to one AHU',
        'n_hot_water_systems':'Number of hot-water systems', 'min_ahus_required':'Minimum AHUs required by the generated structure',
        'floor_plan_shape':'Shape of the floor-plan array', 'zone_map_shape':'Shape of the zone-map array',
        'zone_cells_min':'Minimum grid cells in a zone', 'zone_cells_mean':'Mean grid cells per zone', 'zone_cells_max':'Maximum grid cells in a zone',
        'n_zone_feature_types':'Number of observation feature types per zone', 'expected_zone_elements':'Derived total zone observation elements',
        'n_ahu_feature_types':'Number of observation feature types per AHU', 'expected_ahu_elements':'Derived total AHU observation elements',
        'n_hot_water_feature_types':'Number of hot-water observation feature types', 'expected_hot_water_elements':'Derived hot-water observation elements',
        'n_time_feature_types':'Number of time-indexed observation feature types', 'expected_time_elements':'Derived time-indexed observation elements',
        'n_building_feature_types':'Number of building-wide observation feature types', 'expected_building_elements':'Derived building-wide observation elements',
        'n_feature_types_total':'Total observation feature types', 'expected_vector_length_derived':'Derived total observation vector length'
    }
    return meanings.get(n, f'{n.replace(".", " ").replace("_", " ").capitalize()} (see feature catalog)')

catalog_csv = cat_raw.loc[cat_raw['in_csv'].astype(bool)].set_index('column')
variable_dictionary = pd.DataFrame([
    {'column': c, 'group': catalog_csv.loc[c, 'group'], 'meaning': plain_meaning(c),
     'type': classify_type(c, df_raw[c]), 'catalog_kind': catalog_csv.loc[c, 'kind']}
    for c in df_raw.columns
])
assert len(variable_dictionary) == 54
variable_dictionary

## 3. Minimal preprocessing and quality checks (Member 1)

The source data is already clean: we expect no missing values and no duplicate building rows, so no observations need to be fixed or removed. Keep `df_raw`, `rej_raw`, and `cat_raw` unchanged. In the analysis copy, shorten field names by removing only the first group prefix, then omit constant columns because they cannot define a non-trivial event in this sample. Keep valid outliers, including buildings with up to 305 zones. All probability results on `df` therefore describe accepted buildings and are conditional on the generator accepting them.

In [ ]:
preprocess_log = []
def log_step(name, frame):
    preprocess_log.append({'step': name, 'rows': len(frame), 'columns': frame.shape[1]})
    print(f"{name}: {len(frame)} rows x {frame.shape[1]} columns")

log_step('Raw building file', df_raw)
df = df_raw.copy()
df.columns = [c.split('__', 1)[1] for c in df.columns]
log_step('Shortened group-prefixed names', df)
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
df = df.drop(columns=const_cols)
log_step(f'Dropped {len(const_cols)} constant columns (analysis frame only)', df)

# Quality checks required for the shared data contract.
missing_total = int(df_raw.isna().sum().sum())
duplicate_rows = int(df_raw.duplicated().sum())
assert missing_total == 0
assert duplicate_rows == 0
assert df.seed.is_unique
assert df.arrays_sha256.is_unique
assert (df.n_zones == df.n_vav_boxes).all()
assert (df.n_ahus == df['ahu.num_ahus'] + df.ahus_added_beyond_config).all()

accepted_seeds = set(df.seed.astype(int))
rejected_seed_set = set(rej_raw.seed.astype(int))
expected_seeds = set(range(100000, 100282))
assert accepted_seeds.isdisjoint(rejected_seed_set)
assert accepted_seeds | rejected_seed_set == expected_seeds
assert len(accepted_seeds) == 200 and len(rejected_seed_set) == 82

quality_checks = pd.Series({
    'missing cells in raw buildings': missing_total,
    'duplicate raw rows': duplicate_rows,
    'building seeds unique': df.seed.is_unique,
    'array hashes unique': df.arrays_sha256.is_unique,
    'n_zones equals n_vav_boxes for every row': (df.n_zones == df.n_vav_boxes).all(),
    'n_ahus equals configured plus added for every row': (df.n_ahus == df['ahu.num_ahus'] + df.ahus_added_beyond_config).all(),
    'accepted/rejected seeds disjoint and complete': accepted_seeds.isdisjoint(rejected_seed_set) and accepted_seeds | rejected_seed_set == expected_seeds
}, name='result')
display(quality_checks.to_frame())
display(pd.DataFrame(preprocess_log))
print('Constant columns (kept in df_raw):', const_cols)

## 4. Rejected seeds and shared event columns (Member 1)

Parse the rejection reason into the number of undersized zones and the smallest zone area. Build one row per attempted seed with an `accepted` flag. Publish the following reusable Boolean columns without renaming: `A_large`, `B_extra_ahu`, `F2`, `F3`, `F4`, and `O2`. The large-building threshold is the median zone count (64) and uses a strict greater-than comparison, so ties remain in the not-large group.

In [ ]:
rej = rej_raw.copy()
rej['n_small_zones'] = rej.reason.str.extract(r'produces (\d+) zone')[0].astype(int)
rej['smallest_m2'] = rej.reason.str.extract(r'smallest ([\d.]+) m2')[0].astype(float)
assert rej[['n_small_zones', 'smallest_m2']].notna().all().all()

seeds = pd.DataFrame({'seed': range(100000, 100282)})
seeds['accepted'] = seeds.seed.isin(accepted_seeds)
assert seeds.accepted.sum() == 200 and (~seeds.accepted).sum() == 82

zone_median = df.n_zones.median()
df['A_large'] = df.n_zones > zone_median
df['B_extra_ahu'] = df.ahus_added_beyond_config > 0
df['F2'] = df['floor_plan.num_floors'] == 2
df['F3'] = df['floor_plan.num_floors'] == 3
df['F4'] = df['floor_plan.num_floors'] == 4
df['O2'] = df['simulation.occupants_per_zone'] == 2

event_counts = pd.Series({
    'A_large (n_zones > median)': int(df.A_large.sum()),
    'B_extra_ahu': int(df.B_extra_ahu.sum()),
    'F2': int(df.F2.sum()), 'F3': int(df.F3.sum()), 'F4': int(df.F4.sum()),
    'O2': int(df.O2.sum()),
}, name='count')
display(event_counts.to_frame())
print('Shared event columns:', ['A_large', 'B_extra_ahu', 'F2', 'F3', 'F4', 'O2'])

### Preprocessing notes and limitations (Member 1)

No building rows were removed: the analysis retains all 200 accepted configurations. Constant columns were removed only from `df`, not from the source or `df_raw`; dropping them does not change any event probability. Redundant measures such as zone count and VAV-box count remain available for interpretation but carry the same information in this dataset. The 82 rejected seeds are tracked separately and are not mixed into building-level probability denominators.

The buildings are generator outputs, not a random sample of real buildings. Conclusions are conditional on generator acceptance and this seed range. There is no energy-use or comfort outcome in these files. Observation vector lengths are derived from a schema formula rather than measured for every building in a live rollout. All buildings use the same weather preset, and the sample contains constants and redundant features. The median threshold is a convenient convention, not a physical definition of complexity.

## 5. Probability applications (Members 2-4)

Team members: add each application using the required five-part template: question; events and assumptions; rule and why it fits; calculation and result; interpretation. Keep this section after the shared event-column cell.

## 6. Validation (Member 4)

Add independent validation calculations, Monte Carlo comparison, bootstrap interval, and preprocessing sensitivity analysis here.

## 7. Insights and limitations (Member 5)

Summarize findings with numbers, discuss limitations, and explain relevance to the Smart Buildings competition.